# ⚽ World Cup 2026 — AI Predictor
## Day 1: Exploratory Data Analysis + Feature Engineering

**Project:** Build an ML + LLM system to predict World Cup 2026 match outcomes  
**Author:** Yosmely Bermúdez  
**Stack:** Python · Pandas · Scikit-learn · LangChain · Streamlit  

---

### What we build today

| Step | Description | Output |
|------|-------------|--------|
| 1 | Setup & data loading | All datasets in memory |
| 2 | Historical EDA (1930–2022) | Patterns, trends, anomalies |
| 3 | Elo Rating System | `elo_ratings` per team per year |
| 4 | Qatar 2022 team features | `team_features_2022` |
| 5 | Player Impact Score | `player_impact` per team |
| 6 | Master feature table | `master_features.csv` — model-ready |

---

## 0. Setup & Dependencies

In [ ]:
# ── Install any missing libraries ──────────────────────────────────────────
!pip install -q plotly kaleido

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

print('✅ Libraries ready')

✅ Libraries ready


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Set your folder ID here ───────────────────────────────────────────────
# Paste the ID from your Drive folder URL:
# https://drive.google.com/drive/folders/  <--- THIS PART --->
FOLDER_ID = '1vaJzdwCScXwAl1sHggowEpHhscBvPoAu'

import os
# Find the folder path by searching Drive
# Alternative: set the path directly if you know it
# DATA_PATH = '/content/drive/MyDrive/World Cup Data/Data/'

# Auto-locate folder from ID (works if folder is in your Drive root or shared)
result = !find /content/drive/MyDrive -name '*.csv' 2>/dev/null | head -5
print('Sample CSV files found in Drive:')
for r in result:
    print(' ', r)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sample CSV files found in Drive:
  /content/drive/MyDrive/c17-83-m-data-bi/Datos y prueba MP Retail/tabla_transaccional.csv
  /content/drive/MyDrive/Colab Notebooks/Test Yosmely Bermúdez/movie_metadata.csv
  /content/drive/MyDrive/Colab Notebooks/Test Yosmely Bermúdez/event_data.csv
  /content/drive/MyDrive/Challenge Ticmas/lk_nyc_zones.csv
  /content/drive/MyDrive/encuestas_abiertas.csv


In [ ]:
# ── Configure data path ────────────────────────────────────────────────────
# Option A: Set path manually (recommended — more reliable)
# Change 'WorldCup2026/data' to match your actual folder structure
DATA_PATH = '/content/drive/MyDrive/World Cup Data/Data/'

# Option B: Use folder ID via gdown
# !pip install -q gdown
# import gdown
# gdown.download_folder(id=FOLDER_ID, output='/content/wc_data', quiet=False)
# DATA_PATH = '/content/wc_data/'

# Verify files are accessible
expected_files = [
    'matches_1930_2022.csv',
    'world_cup.csv',
    'FIFA World Cup 1930-2022 All Match Dataset.csv',
    'fifa_ranking_2022-10-06.csv',
    'data (1).csv',
    'team_data.csv',
    'group_stats.csv',
    'player_stats.csv',
    'player_shooting.csv',
    'player_defense.csv',
    'player_passing.csv',
    'player_gca.csv',
    'player_keepers.csv',
    'player_keepersadv.csv',
    'player_misc.csv',
    'player_possession.csv',
    'player_playingtime.csv',
    'player_passing_types.csv',
]

print('🔍 Checking files...')
missing = []
for f in expected_files:
    path = DATA_PATH + f
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    if '❌' in status:
        missing.append(f)
    print(f'  {status}  {f}')

if missing:
    print(f'\n⚠️  {len(missing)} file(s) not found. Check DATA_PATH or file names.')
else:
    print('\n✅ All files found. Ready to load!')

🔍 Checking files...
  ✅  matches_1930_2022.csv
  ✅  world_cup.csv
  ✅  FIFA World Cup 1930-2022 All Match Dataset.csv
  ✅  fifa_ranking_2022-10-06.csv
  ✅  data (1).csv
  ✅  team_data.csv
  ✅  group_stats.csv
  ✅  player_stats.csv
  ✅  player_shooting.csv
  ✅  player_defense.csv
  ✅  player_passing.csv
  ✅  player_gca.csv
  ✅  player_keepers.csv
  ✅  player_keepersadv.csv
  ✅  player_misc.csv
  ✅  player_possession.csv
  ✅  player_playingtime.csv
  ✅  player_passing_types.csv

✅ All files found. Ready to load!


## 1. Load All Datasets

In [ ]:
# ── Helper: load with encoding fallback ───────────────────────────────────
def load_csv(filename, **kwargs):
    """Load CSV with UTF-8 fallback to latin-1."""
    path = DATA_PATH + filename
    try:
        return pd.read_csv(path, encoding='utf-8', **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1', **kwargs)

# ── Load all datasets ─────────────────────────────────────────────────────

# Historical data (1930–2022)
matches      = load_csv('matches_1930_2022.csv')
world_cups   = load_csv('world_cup.csv')
all_matches  = load_csv('FIFA World Cup 1930-2022 All Match Dataset.csv')

# Qatar 2022 — match level
qatar_matches = load_csv('data (1).csv')
team_data     = load_csv('team_data.csv')
group_stats   = load_csv('group_stats.csv')
ranking_2022  = load_csv('fifa_ranking_2022-10-06.csv')

# Qatar 2022 — player level
p_stats    = load_csv('player_stats.csv')
p_shooting = load_csv('player_shooting.csv')
p_defense  = load_csv('player_defense.csv')
p_passing  = load_csv('player_passing.csv')
p_gca      = load_csv('player_gca.csv')
p_keepers  = load_csv('player_keepers.csv')
p_keepadv  = load_csv('player_keepersadv.csv')
p_misc     = load_csv('player_misc.csv')
p_possess  = load_csv('player_possession.csv')
p_playtime = load_csv('player_playingtime.csv')
p_passtypes= load_csv('player_passing_types.csv')

print('✅ All datasets loaded')
print(f'\n{'Dataset':<35} {'Shape':<15} Description')
print('-' * 75)
datasets = [
    (matches,       'matches_1930_2022',    'All WC matches with detailed stats'),
    (world_cups,    'world_cups',           'WC summary: hosts, champions, top scorers'),
    (qatar_matches, 'qatar_matches',        'Qatar 2022: 64 matches, 53 features'),
    (team_data,     'team_data',            'Qatar 2022: 32 teams, 189 features'),
    (group_stats,   'group_stats',          'Qatar 2022: group stage standings'),
    (ranking_2022,  'ranking_2022',         'FIFA ranking at WC 2022 start'),
    (p_stats,       'player_stats',         'Player summary stats (680 players)'),
]
for df, name, desc in datasets:
    print(f'{name:<35} {str(df.shape):<15} {desc}')

✅ All datasets loaded

Dataset                             Shape           Description
---------------------------------------------------------------------------
matches_1930_2022                   (964, 44)       All WC matches with detailed stats
world_cups                          (22, 9)         WC summary: hosts, champions, top scorers
qatar_matches                       (64, 53)        Qatar 2022: 64 matches, 53 features
team_data                           (32, 189)       Qatar 2022: 32 teams, 189 features
group_stats                         (32, 16)        Qatar 2022: group stage standings
ranking_2022                        (211, 7)        FIFA ranking at WC 2022 start
player_stats                        (680, 31)       Player summary stats (680 players)


## 2. Historical EDA (1930–2022)

We extract patterns that will become **features** for our model:
- Which teams win the most?
- Home/away advantage in World Cups?
- Goal trends across decades?
- Which rounds are most unpredictable?

In [ ]:
# ── Basic shape & quality check ────────────────────────────────────────────
print('=== matches_1930_2022 ===')
print(f'Shape: {matches.shape}')
print(f'Years covered: {sorted(matches["Year"].unique())}')
print(f'\nNull values (top cols):')
key_cols = ['home_team','away_team','home_score','away_score','home_xg','away_xg','Year','Round']
print(matches[key_cols].isnull().sum())
print(f'\nxG available from year: {matches[matches["home_xg"].notna()]["Year"].min()}')

=== matches_1930_2022 ===
Shape: (964, 44)
Years covered: [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]

Null values (top cols):
home_team       0
away_team       0
home_score      0
away_score      0
home_xg       836
away_xg       836
Year            0
Round           0
dtype: int64

xG available from year: 2018


In [ ]:
# ── Determine match outcomes ───────────────────────────────────────────────
# We need a clean result column: home_win / draw / away_win

def get_result(row):
    """Return outcome from home team perspective."""
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

matches['result'] = matches.apply(get_result, axis=1)
matches['total_goals'] = matches['home_score'] + matches['away_score']

# Result distribution
result_counts = matches['result'].value_counts()
print('Overall result distribution:')
print(result_counts)
print(f'\nHome win rate: {result_counts["home_win"] / len(matches):.1%}')
print(f'Draw rate:     {result_counts["draw"] / len(matches):.1%}')
print(f'Away win rate: {result_counts["away_win"] / len(matches):.1%}')
print(f'\nAvg goals per match: {matches["total_goals"].mean():.2f}')

Overall result distribution:
result
home_win    532
away_win    218
draw        214
Name: count, dtype: int64

Home win rate: 55.2%
Draw rate:     22.2%
Away win rate: 22.6%

Avg goals per match: 2.82


In [ ]:
# ── Goals per match over time ──────────────────────────────────────────────
goals_by_year = matches.groupby('Year').agg(
    avg_goals=('total_goals', 'mean'),
    total_matches=('total_goals', 'count')
).reset_index()

fig = px.line(
    goals_by_year,
    x='Year', y='avg_goals',
    markers=True,
    title='Average Goals per Match — World Cup 1930–2022',
    labels={'avg_goals': 'Avg Goals / Match', 'Year': 'Year'},
    template='plotly_white'
)
fig.add_hline(y=goals_by_year['avg_goals'].mean(),
              line_dash='dash', line_color='red',
              annotation_text=f'Historical avg: {goals_by_year["avg_goals"].mean():.2f}')
fig.show()

print('💡 Note: The xG era starts from 2018 — earlier matches only have actual goals.')

💡 Note: The xG era starts from 2018 — earlier matches only have actual goals.


In [ ]:
# ── All-time win rankings ──────────────────────────────────────────────────
# Build a wins table from both home and away perspectives

# Home wins
home_wins = matches[matches['result'] == 'home_win'].groupby('home_team').size().reset_index()
home_wins.columns = ['team', 'wins_as_home']

# Away wins
away_wins = matches[matches['result'] == 'away_win'].groupby('away_team').size().reset_index()
away_wins.columns = ['team', 'wins_as_away']

# Total matches played
home_played = matches.groupby('home_team').size().reset_index()
home_played.columns = ['team', 'home_played']
away_played = matches.groupby('away_team').size().reset_index()
away_played.columns = ['team', 'away_played']

# Merge everything
win_table = pd.merge(home_wins, away_wins, on='team', how='outer').fillna(0)
win_table = pd.merge(win_table, home_played, on='team', how='outer').fillna(0)
win_table = pd.merge(win_table, away_played, on='team', how='outer').fillna(0)

win_table['total_wins']    = win_table['wins_as_home'] + win_table['wins_as_away']
win_table['total_matches'] = win_table['home_played'] + win_table['away_played']
win_table['win_rate']      = win_table['total_wins'] / win_table['total_matches']

# Keep only teams with 5+ matches (filter noise)
win_table = win_table[win_table['total_matches'] >= 5].sort_values('total_wins', ascending=False)

print('Top 15 teams — all-time WC wins:')
print(win_table[['team','total_wins','total_matches','win_rate']].head(15).to_string(index=False))

Top 15 teams — all-time WC wins:
        team  total_wins  total_matches  win_rate
      Brazil      76.000        114.000     0.667
   Argentina      47.000         88.000     0.534
       Italy      45.000         83.000     0.542
      France      39.000         73.000     0.534
     Germany      37.000         56.000     0.661
     England      32.000         74.000     0.432
       Spain      31.000         67.000     0.463
West Germany      31.000         56.000     0.554
 Netherlands      30.000         55.000     0.545
     Uruguay      25.000         59.000     0.424
     Belgium      21.000         51.000     0.412
      Sweden      19.000         51.000     0.373
      Poland      17.000         38.000     0.447
      Mexico      17.000         60.000     0.283
    Portugal      17.000         35.000     0.486


In [ ]:
# ── Visualize top 15 win rates (min 10 matches) ────────────────────────────
top_teams = win_table[win_table['total_matches'] >= 10].head(20).sort_values('win_rate')

fig = px.bar(
    top_teams,
    x='win_rate', y='team',
    orientation='h',
    color='win_rate',
    color_continuous_scale='Blues',
    title='Win Rate at World Cups — Top 20 Teams (min 10 matches)',
    labels={'win_rate': 'Win Rate', 'team': ''},
    text=top_teams['total_wins'].astype(int).astype(str) + ' W',  # ← fix: use top_teams, not win_table
    template='plotly_white'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# ── Results by round (group stage vs knockout) ─────────────────────────────
round_results = matches.groupby('Round')['result'].value_counts(normalize=True).unstack().fillna(0)

# Simplify round names
round_map = {
    'Group stage': 'Group stage',
    'Round of 16': 'Round of 16',
    'Quarter-finals': 'Quarter-finals',
    'Semi-finals': 'Semi-finals',
    'Final': 'Final',
    'Third-place match': 'Third place'
}

# Filter to main rounds
key_rounds = [r for r in round_results.index if r in round_map]
round_filtered = round_results.loc[[r for r in round_map if r in round_results.index]]

print('Result distribution by round (normalized):')
print(round_filtered.round(3).to_string())

print('\n💡 Insight: Knockout rounds have ~0% draws (goes to extra time/penalties)')
print('   Group stage draw rate is much higher — important for our model!')

Result distribution by round (normalized):
result             away_win  draw  home_win
Round                                      
Group stage           0.259 0.237     0.504
Round of 16           0.165 0.155     0.680
Quarter-finals        0.186 0.229     0.586
Semi-finals           0.237 0.132     0.632
Final                 0.190 0.143     0.667
Third-place match     0.250 0.000     0.750

💡 Insight: Knockout rounds have ~0% draws (goes to extra time/penalties)
   Group stage draw rate is much higher — important for our model!


In [ ]:
# ── Head-to-head matrix for top teams ────────────────────────────────────
top_10_teams = win_table.head(10)['team'].tolist()

def h2h_wins(team_a, team_b, df):
    """Count wins of team_a vs team_b in World Cups."""
    home = len(df[(df['home_team'] == team_a) & (df['away_team'] == team_b) & (df['result'] == 'home_win')])
    away = len(df[(df['away_team'] == team_a) & (df['home_team'] == team_b) & (df['result'] == 'away_win')])
    return home + away

h2h_matrix = pd.DataFrame(index=top_10_teams, columns=top_10_teams, dtype=float)
for t1 in top_10_teams:
    for t2 in top_10_teams:
        if t1 == t2:
            h2h_matrix.loc[t1, t2] = np.nan
        else:
            h2h_matrix.loc[t1, t2] = h2h_wins(t1, t2, matches)

fig = px.imshow(
    h2h_matrix.astype(float),
    text_auto=True,
    color_continuous_scale='Blues',
    title='Head-to-Head Wins — Top 10 WC Teams (row beats column)',
    template='plotly_white'
)
fig.show()
print('Row = winner, Column = loser. Value = number of WC victories.')

Row = winner, Column = loser. Value = number of WC victories.


## 3. Elo Rating System

Elo ratings are the **most predictive single feature** in football forecasting models.
We compute Elo for every team across all 964 historical matches.

**How it works:**
- Every team starts at 1500
- After each match, points transfer from loser to winner
- The K-factor controls how fast ratings change
- The **expected score** determines how many points are transferred

In [ ]:
# ── Elo Rating Engine ─────────────────────────────────────────────────────

def expected_score(rating_a, rating_b):
    """Probability that team A beats team B."""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=32):
    """Update Elo rating after a match."""
    return rating + k * (actual - expected)

def compute_elo_ratings(matches_df, k=32, initial=1500):
    """
    Compute Elo ratings for all teams across all matches.

    Returns:
        - elo_dict: final ratings per team
        - elo_history: DataFrame with rating after each match
    """
    elo = {}           # team -> current rating
    history = []       # list of (year, match_idx, team, elo_before, elo_after)

    # Sort by Year (chronological order)
    df = matches_df.sort_values('Year').copy()

    for _, row in df.iterrows():
        home = row['home_team']
        away = row['away_team']
        year = row['Year']

        # Initialize new teams
        if home not in elo:
            elo[home] = initial
        if away not in elo:
            elo[away] = initial

        r_home = elo[home]
        r_away = elo[away]

        # Expected scores
        exp_home = expected_score(r_home, r_away)
        exp_away = 1 - exp_home

        # Actual scores (1=win, 0.5=draw, 0=loss)
        result = row['result']
        if result == 'home_win':
            act_home, act_away = 1.0, 0.0
        elif result == 'away_win':
            act_home, act_away = 0.0, 1.0
        else:
            act_home, act_away = 0.5, 0.5

        # Update ratings
        new_home = update_elo(r_home, exp_home, act_home, k)
        new_away = update_elo(r_away, exp_away, act_away, k)

        # Record history
        history.append({'year': year, 'team': home, 'elo_before': r_home, 'elo_after': new_home})
        history.append({'year': year, 'team': away, 'elo_before': r_away, 'elo_after': new_away})

        elo[home] = new_home
        elo[away] = new_away

    history_df = pd.DataFrame(history)
    return elo, history_df

# Compute Elo (requires 'result' column from Section 2)
elo_final, elo_history = compute_elo_ratings(matches, k=32)

# Convert to DataFrame
elo_df = pd.DataFrame.from_dict(elo_final, orient='index', columns=['elo_rating'])
elo_df = elo_df.sort_values('elo_rating', ascending=False).reset_index()
elo_df.columns = ['team', 'elo_rating']

print('Top 20 teams by Elo rating (based on all WC matches 1930–2022):')
print(elo_df.head(20).to_string(index=False))

Top 20 teams by Elo rating (based on all WC matches 1930–2022):
        team  elo_rating
 Netherlands    1704.439
      Brazil    1692.769
      France    1692.645
     Germany    1667.371
   Argentina    1655.619
West Germany    1651.873
       Italy    1613.168
       Spain    1591.768
     Belgium    1590.027
     England    1582.902
     Croatia    1575.701
    Portugal    1557.479
     Uruguay    1548.446
Soviet Union    1545.665
      Sweden    1539.195
  Yugoslavia    1538.740
     Türkiye    1535.654
     Romania    1526.985
     Senegal    1526.114
    Colombia    1525.775


In [ ]:
# ── Elo trajectory for top 6 teams ────────────────────────────────────────
top_6 = elo_df.head(6)['team'].tolist()

# Get last Elo per team per year
elo_by_year = elo_history.groupby(['year', 'team'])['elo_after'].last().reset_index()
elo_top6 = elo_by_year[elo_by_year['team'].isin(top_6)]

fig = px.line(
    elo_top6,
    x='year', y='elo_after', color='team',
    markers=True,
    title='Elo Rating Trajectory — Top 6 WC Teams (1930–2022)',
    labels={'elo_after': 'Elo Rating', 'year': 'Year'},
    template='plotly_white'
)
fig.show()

print('\n💡 This is FEATURE 1 for our model: elo_rating at the start of each WC')


💡 This is FEATURE 1 for our model: elo_rating at the start of each WC


In [ ]:
# ── Build Elo feature table: one row per (team, year) ─────────────────────
# For each team + WC year, get their Elo BEFORE that tournament starts

wc_years = sorted(matches['Year'].unique())
elo_features = []

for year in wc_years:
    # Get all teams that participated this year
    year_teams = set(
        matches[matches['Year'] == year]['home_team'].tolist() +
        matches[matches['Year'] == year]['away_team'].tolist()
    )

    # Get Elo at START of this tournament (last rating before this year)
    for team in year_teams:
        team_history = elo_history[
            (elo_history['team'] == team) &
            (elo_history['year'] < year)
        ]

        if len(team_history) > 0:
            elo_entry = team_history.iloc[-1]['elo_after']
        else:
            elo_entry = 1500  # No prior WC history

        elo_features.append({
            'team': team,
            'year': year,
            'elo_entry': elo_entry
        })

elo_feature_df = pd.DataFrame(elo_features)
print(f'Elo feature table shape: {elo_feature_df.shape}')
print(f'\nSample — Argentina Elo entry per WC:')
print(elo_feature_df[elo_feature_df['team'] == 'Argentina'].to_string(index=False))

Elo feature table shape: (489, 3)

Sample — Argentina Elo entry per WC:
     team  year  elo_entry
Argentina  1930   1500.000
Argentina  1934   1542.341
Argentina  1958   1523.710
Argentina  1962   1502.319
Argentina  1966   1503.833
Argentina  1974   1516.680
Argentina  1978   1493.296
Argentina  1982   1566.748
Argentina  1986   1545.140
Argentina  1990   1624.844
Argentina  1994   1614.264
Argentina  1998   1587.436
Argentina  2002   1602.374
Argentina  2006   1594.472
Argentina  2010   1627.759
Argentina  2014   1649.759
Argentina  2018   1677.293
Argentina  2022   1637.150


## 4. Qatar 2022 Team Features

We extract the most predictive features from `team_data` and `group_stats`.  
These will serve as **reference signals** for estimating 2026 team strength.

In [ ]:
# ── Explore team_data (189 columns!) ──────────────────────────────────────
print(f'team_data shape: {team_data.shape}')
print(f'\nColumn groups:')
cols = team_data.columns.tolist()

# Categorize columns
categories = {
    'basic':     [c for c in cols if c in ['team','possession','avg_age','games']],
    'goals/xG':  [c for c in cols if 'goal' in c.lower() or 'xg' in c.lower()],
    'passing':   [c for c in cols if 'pass' in c.lower()],
    'shooting':  [c for c in cols if 'shot' in c.lower() or 'shoot' in c.lower()],
    'defense':   [c for c in cols if 'tackle' in c.lower() or 'intercept' in c.lower() or 'clear' in c.lower()],
    'pressing':  [c for c in cols if 'press' in c.lower()],
}

for cat, cat_cols in categories.items():
    print(f'  {cat}: {len(cat_cols)} cols → {cat_cols[:5]}')

team_data shape: (32, 189)

Column groups:
  basic: 4 cols → ['team', 'avg_age', 'possession', 'games']
  goals/xG: 40 cols → ['goals', 'goals_pens', 'goals_per90', 'goals_assists_per90', 'goals_pens_per90']
  passing: 38 cols → ['gk_passes_completed_launched', 'gk_passes_launched', 'gk_passes_pct_launched', 'gk_passes', 'gk_passes_throws']
  shooting: 16 cols → ['gk_shots_on_target_against', 'gk_psnpxg_per_shot_on_target_against', 'shots', 'shots_on_target', 'shots_on_target_pct']
  defense: 10 cols → ['tackles', 'tackles_won', 'tackles_def_3rd', 'tackles_mid_3rd', 'tackles_att_3rd']
  pressing: 0 cols → []


In [ ]:
# ── Select high-signal team features ─────────────────────────────────────
# Chosen based on football analytics research — most predictive features

TEAM_FEATURES = [
    'team',
    # Possession & Style
    'possession',
    'avg_age',
    # Attacking
    'goals',
    'xg',                    # Expected goals scored
    'npxg',                  # Non-penalty xG (purer measure)
    'shots',
    'shots_on_target',
    # Progressive play
    'progressive_passes',
    'progressive_carries',
    # Passing quality
    'passes_completed',
    'passes_pct',
    # Defensive
    'tackles',
    'tackles_won',
    'interceptions',
    'clearances',
    # Pressing
    'pressures',
    'pressure_regains',
    # Set pieces
    'corner_kicks',
    'free_kicks',
]

# Keep only columns that exist (some may differ by dataset version)
available = [c for c in TEAM_FEATURES if c in team_data.columns]
missing_cols = [c for c in TEAM_FEATURES if c not in team_data.columns]

print(f'Features available: {len(available)}/{len(TEAM_FEATURES)}')
if missing_cols:
    print(f'Not found (check column names): {missing_cols}')

team_features = team_data[available].copy()
print(f'\nteam_features shape: {team_features.shape}')
print(team_features.head(5).to_string())

Features available: 16/20
Not found (check column names): ['progressive_carries', 'pressures', 'pressure_regains', 'free_kicks']

team_features shape: (32, 16)
        team  possession  avg_age  goals     xg   npxg  shots  shots_on_target  progressive_passes  passes_completed  passes_pct  tackles  tackles_won  interceptions  clearances  corner_kicks
0  Argentina      57.400   28.400     15 15.200 11.400     95               41                 217              3911      84.600      123           69             52         124            39
1  Australia      37.800   28.700      3  2.300  2.300     26                8                  68              1254      73.900       58           30             40         104             8
2    Belgium      57.000   30.600      1  4.700  4.700     35                9                  99              1598      84.800       48           27             17          59            17
3     Brazil      56.200   28.500      8 12.000 11.200     95           

In [ ]:
# ── Merge with group stage results ────────────────────────────────────────
# group_stats has: wins, draws, losses, goals_scored, xG, points

group_clean = group_stats[['team', 'wins', 'draws', 'losses',
                             'goals_scored', 'goals_against', 'goal_difference',
                             'points', 'expected_goal_scored', 'exp_goal_conceded',
                             'exp_goal_difference']].copy()
group_clean.columns = [
    'team', 'group_wins', 'group_draws', 'group_losses',
    'group_gf', 'group_ga', 'group_gd',
    'group_pts', 'group_xgf', 'group_xga', 'group_xgd'
]

# Merge FIFA ranking
ranking_clean = ranking_2022[['team', 'rank', 'points']].copy()
ranking_clean.columns = ['team', 'fifa_rank', 'fifa_points']

# Combine all Qatar 2022 team features
qatar_team_features = team_features.copy()
qatar_team_features = pd.merge(qatar_team_features, group_clean, on='team', how='left')
qatar_team_features = pd.merge(qatar_team_features, ranking_clean, on='team', how='left')

# Add Elo at start of Qatar 2022
elo_2022 = elo_feature_df[elo_feature_df['year'] == 2022][['team', 'elo_entry']]
elo_2022.columns = ['team', 'elo_2022']
qatar_team_features = pd.merge(qatar_team_features, elo_2022, on='team', how='left')

print(f'Qatar 2022 team feature table: {qatar_team_features.shape}')
print(f'\nTop 10 by Elo 2022:')
cols_show = ['team', 'elo_2022', 'fifa_rank', 'group_pts', 'group_xgd']
cols_show = [c for c in cols_show if c in qatar_team_features.columns]
print(qatar_team_features.sort_values('elo_2022', ascending=False)[cols_show].head(10).to_string(index=False))

Qatar 2022 team feature table: (32, 29)

Top 10 by Elo 2022:
       team  elo_2022  fifa_rank  group_pts  group_xgd
     Brazil  1707.271      1.000      6.000      4.900
Netherlands  1693.440      8.000      7.000     -0.300
    Germany  1688.083     11.000      4.000      6.700
     France  1678.838      4.000      6.000      5.500
  Argentina  1637.150      3.000      6.000      5.200
      Spain  1605.481      7.000      4.000      2.500
    Belgium  1605.317      2.000      4.000      0.200
    Croatia  1564.082     12.000      5.000      0.000
    England  1563.973      5.000      7.000      2.900
    Uruguay  1556.778     14.000      4.000      0.100


In [ ]:
# ── Correlation heatmap — which features predict group stage success? ─────
if 'group_pts' in qatar_team_features.columns:
    numeric_cols = qatar_team_features.select_dtypes(include='number').columns.tolist()
    corr_with_pts = qatar_team_features[numeric_cols].corr()['group_pts'].drop('group_pts')
    corr_with_pts = corr_with_pts.dropna().sort_values(key=abs, ascending=False).head(20)

    fig = px.bar(
        x=corr_with_pts.values,
        y=corr_with_pts.index,
        orientation='h',
        title='Feature Correlation with Group Stage Points (Qatar 2022)',
        labels={'x': 'Pearson Correlation', 'y': 'Feature'},
        color=corr_with_pts.values,
        color_continuous_scale='RdBu',
        template='plotly_white'
    )
    fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
    fig.show()

    print('\n💡 These are your MOST IMPORTANT features for the model!')
    print('   Top 5 correlated features:', corr_with_pts.abs().head(5).index.tolist())


💡 These are your MOST IMPORTANT features for the model!
   Top 5 correlated features: ['group_wins', 'group_losses', 'group_gd', 'group_ga', 'goals']


## 5. Player Impact Score (PIS)

**This is the differentiator of our project.** Most WC prediction models only use team-level stats.
We build a composite score per team that captures individual player quality.

**PIS Formula (per 90 min, normalized):**
- Attacking: `goals_per90 * 0.25 + xg_per90 * 0.20 + xg_assist_per90 * 0.15`
- Shot creation: `sca_per90 * 0.10`  
- Defensive: `tackles_won_per90 * 0.15 + interceptions_per90 * 0.15`

Then we **aggregate to team level** using top-N players by minutes played.

In [ ]:
# ── Merge player datasets ─────────────────────────────────────────────────
# Base: player_stats (has minutes, goals, assists, xG)
# Add: shooting (shots, xG detail)
# Add: gca (shot-creating actions — creativity proxy)
# Add: defense (tackles, interceptions)

PLAYER_KEY = ['player', 'team']

# Clean and prepare each dataset for merge
def clean_player_df(df, suffix=''):
    """Keep player+team as join keys, add suffix to avoid col collisions."""
    numeric = df.select_dtypes(include='number').columns.tolist()
    cols_keep = PLAYER_KEY + numeric
    cols_keep = [c for c in cols_keep if c in df.columns]
    result = df[cols_keep].copy()
    if suffix:
        rename = {c: f'{c}{suffix}' for c in numeric if c not in PLAYER_KEY}
        result = result.rename(columns=rename)
    return result

# Merge player stats
player_merged = p_stats[['player', 'team', 'position', 'age', 'minutes_90s',
                           'goals', 'assists', 'goals_per90', 'assists_per90',
                           'xg', 'xg_assist', 'xg_per90', 'xg_assist_per90']].copy()

# Add shooting stats
shoot_cols = ['player', 'team', 'shots', 'shots_on_target', 'shots_on_target_pct',
              'shots_per90', 'goals_per_shot_on_target', 'npxg']
shoot_cols = [c for c in shoot_cols if c in p_shooting.columns]
player_merged = pd.merge(player_merged, p_shooting[shoot_cols], on=PLAYER_KEY, how='left')

# Add goal-creating actions (creativity)
gca_cols = ['player', 'team', 'sca', 'sca_per90', 'gca', 'gca_per90']
gca_cols = [c for c in gca_cols if c in p_gca.columns]
player_merged = pd.merge(player_merged, p_gca[gca_cols], on=PLAYER_KEY, how='left')

# Add defense
def_cols = ['player', 'team', 'tackles', 'tackles_won', 'interceptions',
            'blocks', 'clearances']
def_cols = [c for c in def_cols if c in p_defense.columns]
player_merged = pd.merge(player_merged, p_defense[def_cols], on=PLAYER_KEY, how='left')

print(f'Player merged shape: {player_merged.shape}')
print(f'Columns: {player_merged.columns.tolist()[:20]}')

Player merged shape: (680, 28)
Columns: ['player', 'team', 'position', 'age', 'minutes_90s', 'goals', 'assists', 'goals_per90', 'assists_per90', 'xg', 'xg_assist', 'xg_per90', 'xg_assist_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'goals_per_shot_on_target', 'npxg', 'sca']


In [ ]:
# ── Compute Player Impact Score ───────────────────────────────────────────
# Filter: minimum 90 minutes played (reduces noise from 5-min substitutes)
players = player_merged[player_merged['minutes_90s'] >= 1.0].copy()

# Normalize per 90 min where raw counts exist
per90_base = players['minutes_90s'].clip(lower=0.1)  # avoid div by zero

# Safe division helper
def safe_per90(col):
    return players[col].fillna(0) / per90_base if col in players.columns else pd.Series(0, index=players.index)

# Build PIS components
players['pis_attack'] = (
    players['goals_per90'].fillna(0) * 0.25 +
    players['xg_per90'].fillna(0) * 0.20 +
    players['xg_assist_per90'].fillna(0) * 0.15
)

players['pis_creation'] = (
    players['sca_per90'].fillna(0) * 0.10
    if 'sca_per90' in players.columns else 0
)

players['pis_defense'] = (
    safe_per90('tackles_won') * 0.15 +
    safe_per90('interceptions') * 0.15
)

players['player_impact_score'] = (
    players['pis_attack'] +
    players['pis_creation'] +
    players['pis_defense']
)

# Normalize PIS to 0–100 scale
pis_min = players['player_impact_score'].min()
pis_max = players['player_impact_score'].max()
players['pis_normalized'] = (
    (players['player_impact_score'] - pis_min) / (pis_max - pis_min) * 100
)

print('Top 20 players by Player Impact Score (PIS):')
top_players = players.nlargest(20, 'pis_normalized')[
    ['player', 'team', 'position', 'minutes_90s', 'goals', 'assists',
     'xg_per90', 'pis_normalized']
].round(2)
print(top_players.to_string(index=False))

Top 20 players by Player Impact Score (PIS):
                player           team position  minutes_90s  goals  assists  xg_per90  pis_normalized
           Kai Havertz        Germany       FW        1.100      2        0     1.190         100.000
         Jack Grealish        England       FW        1.000      1        0     0.690          97.440
Giorgian De Arrascaeta        Uruguay       MF        1.200      2        0     1.010          87.830
     Rodrigo Bentancur        Uruguay       MF        2.400      0        0     0.260          84.150
     Eduardo Camavinga         France       DF        1.600      0        0     0.000          82.200
     Juan Pablo Vargas     Costa Rica       DF        1.000      1        0     0.460          81.350
           Rafael Leão       Portugal       FW        1.000      2        0     0.420          80.690
               Rodrygo         Brazil       MF        2.000      0        1     0.260          79.870
         Kylian Mbappé         France

In [ ]:
# ── Aggregate PIS to team level ───────────────────────────────────────────
# We use top-11 players by minutes played (the "starting XI")

def team_pis(team_players, n_top=11):
    """Compute team-level PIS from top N players by minutes."""
    top_n = team_players.nlargest(n_top, 'minutes_90s')
    return {
        'pis_mean':   top_n['pis_normalized'].mean(),
        'pis_max':    top_n['pis_normalized'].max(),
        'pis_sum':    top_n['pis_normalized'].sum(),
        'pis_top3':   top_n.nlargest(3, 'pis_normalized')['pis_normalized'].mean(),
        'avg_xg_p90': top_n['xg_per90'].mean() if 'xg_per90' in top_n else 0,
        'n_players':  len(top_n)
    }

team_pis_df = (
    players
    .groupby('team')
    .apply(team_pis)
    .apply(pd.Series)
    .reset_index()
)

print('Team-level Player Impact Scores:')
print(team_pis_df.sort_values('pis_mean', ascending=False).round(2).to_string(index=False))

Team-level Player Impact Scores:
          team  pis_mean  pis_max  pis_sum  pis_top3  avg_xg_p90  n_players
       Germany    42.030   69.860  462.380    66.230       0.230     11.000
        France    40.990   77.100  450.860    65.600       0.200     11.000
        Brazil    39.150   66.470  430.650    58.580       0.250     11.000
      Cameroon    34.370   60.850  378.100    56.690       0.150     11.000
     Argentina    33.930   71.900  373.220    56.630       0.190     11.000
        Mexico    33.570   59.170  369.270    53.040       0.120     11.000
       England    32.770   69.990  360.480    55.200       0.140     11.000
       IR Iran    32.670   73.790  359.350    55.890       0.130     11.000
       Uruguay    30.930   84.150  340.260    63.170       0.070     11.000
       Tunisia    30.640   52.790  337.000    48.420       0.080     11.000
Korea Republic    30.600   70.570  336.570    48.590       0.090     11.000
         Spain    29.840   53.360  328.260    47.810   

In [ ]:
# ── PIS Visualization ─────────────────────────────────────────────────────
fig = px.scatter(
    team_pis_df.sort_values('pis_mean', ascending=False),
    x='pis_mean', y='pis_top3',
    text='team',
    size='pis_sum',
    color='pis_max',
    color_continuous_scale='Blues',
    title='Team Player Impact Score — Qatar 2022<br>x=avg PIS top11, y=avg PIS top3, size=total PIS',
    labels={'pis_mean': 'Avg PIS (Starting XI)', 'pis_top3': 'Avg PIS (Top 3 Players)'},
    template='plotly_white'
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.show()

print('\n💡 Teams in top-right = strong top players AND strong squad depth')
print('   This bubble chart is perfect for a LinkedIn post!')


💡 Teams in top-right = strong top players AND strong squad depth
   This bubble chart is perfect for a LinkedIn post!


## 6. Build Master Feature Table

We combine everything into a **single match-level DataFrame** ready for the ML model.

Each row = one match  
Features = team A stats − team B stats (difference encoding)  
Target = result (home_win / draw / away_win)

In [ ]:
# ── Build per-match feature table from historical matches ─────────────────

# Merge Elo into matches
matches_feat = matches.copy()

# Add Elo entry for home team
matches_feat = pd.merge(
    matches_feat,
    elo_feature_df.rename(columns={'team': 'home_team', 'elo_entry': 'elo_home', 'year': 'Year'}),
    on=['home_team', 'Year'],
    how='left'
)

# Add Elo entry for away team
matches_feat = pd.merge(
    matches_feat,
    elo_feature_df.rename(columns={'team': 'away_team', 'elo_entry': 'elo_away', 'year': 'Year'}),
    on=['away_team', 'Year'],
    how='left'
)

# Elo difference (most important single feature)
matches_feat['elo_diff'] = matches_feat['elo_home'] - matches_feat['elo_away']

# Add win rates from historical table
win_rate_map = win_table.set_index('team')['win_rate'].to_dict()
matches_feat['winrate_home'] = matches_feat['home_team'].map(win_rate_map).fillna(0.33)
matches_feat['winrate_away'] = matches_feat['away_team'].map(win_rate_map).fillna(0.33)
matches_feat['winrate_diff'] = matches_feat['winrate_home'] - matches_feat['winrate_away']

# Round type encoding (group stage = 0, knockout = 1)
knockout_rounds = ['Round of 16', 'Quarter-finals', 'Semi-finals', 'Final', 'Third-place match']
matches_feat['is_knockout'] = matches_feat['Round'].isin(knockout_rounds).astype(int)

# Tournament era (proxy for football evolution)
matches_feat['era'] = pd.cut(
    matches_feat['Year'],
    bins=[1929, 1966, 1990, 2006, 2022],
    labels=['early', 'golden', 'modern', 'contemporary']
).astype(str)

# Target encoding
target_map = {'home_win': 0, 'draw': 1, 'away_win': 2}
matches_feat['target'] = matches_feat['result'].map(target_map)

print(f'Match feature table shape: {matches_feat.shape}')
print(f'\nKey features created:')
key_feats = ['elo_diff', 'elo_home', 'elo_away', 'winrate_diff', 'is_knockout', 'era', 'target']
print(matches_feat[['home_team', 'away_team', 'Year'] + key_feats].head(10).to_string(index=False))

Match feature table shape: (964, 55)

Key features created:
  home_team   away_team  Year  elo_diff  elo_home  elo_away  winrate_diff  is_knockout          era  target
  Argentina      France  2022   -41.688  1637.150  1678.838        -0.000            1 contemporary       1
    Croatia     Morocco  2022   129.077  1564.082  1435.006         0.216            1 contemporary       0
     France     Morocco  2022   243.832  1678.838  1435.006         0.317            1 contemporary       0
  Argentina     Croatia  2022    73.068  1637.150  1564.082         0.101            1 contemporary       0
    Morocco    Portugal  2022  -117.849  1435.006  1552.855        -0.268            1 contemporary       0
    England      France  2022  -114.865  1563.973  1678.838        -0.102            1 contemporary       2
    Croatia      Brazil  2022  -143.188  1564.082  1707.271        -0.233            1 contemporary       1
Netherlands   Argentina  2022    56.289  1693.440  1637.150         0.011   

In [ ]:
# ── Final feature selection for ML ────────────────────────────────────────
ML_FEATURES = [
    'elo_diff',        # ★ Most predictive — Elo difference
    'elo_home',        # Absolute Elo of home team
    'elo_away',        # Absolute Elo of away team
    'winrate_diff',    # Historical win rate differential
    'winrate_home',
    'winrate_away',
    'is_knockout',     # Stage type matters significantly
]

# Add xG features where available (2018+ matches)
if 'home_xg' in matches_feat.columns:
    matches_feat['xg_diff'] = matches_feat['home_xg'].fillna(0) - matches_feat['away_xg'].fillna(0)
    ML_FEATURES.append('xg_diff')

# Build clean ML dataset
ML_FEATURES_AVAIL = [f for f in ML_FEATURES if f in matches_feat.columns]

ml_dataset = matches_feat[['home_team', 'away_team', 'Year', 'Round'] +
                            ML_FEATURES_AVAIL + ['target']].dropna(
    subset=['elo_diff', 'target']
)

print(f'ML-ready dataset: {ml_dataset.shape}')
print(f'Features: {ML_FEATURES_AVAIL}')
print(f'\nTarget distribution:')
print(ml_dataset['target'].value_counts().rename({0:'home_win', 1:'draw', 2:'away_win'}))
print(f'\nMissing values:')
print(ml_dataset.isnull().sum())

ML-ready dataset: (964, 13)
Features: ['elo_diff', 'elo_home', 'elo_away', 'winrate_diff', 'winrate_home', 'winrate_away', 'is_knockout', 'xg_diff']

Target distribution:
target
home_win    532
away_win    218
draw        214
Name: count, dtype: int64

Missing values:
home_team       0
away_team       0
Year            0
Round           0
elo_diff        0
elo_home        0
elo_away        0
winrate_diff    0
winrate_home    0
winrate_away    0
is_knockout     0
xg_diff         0
target          0
dtype: int64


In [ ]:
# ── Save master feature table ─────────────────────────────────────────────
OUTPUT_PATH = DATA_PATH  # Save back to same Drive folder

# 1. ML-ready match dataset (historical)
ml_dataset.to_csv(OUTPUT_PATH + 'ml_matches_features.csv', index=False)
print('✅ Saved: ml_matches_features.csv')

# 2. Elo ratings (final + per-year)
elo_df.to_csv(OUTPUT_PATH + 'elo_final_ratings.csv', index=False)
print('✅ Saved: elo_final_ratings.csv')

elo_feature_df.to_csv(OUTPUT_PATH + 'elo_entry_per_year.csv', index=False)
print('✅ Saved: elo_entry_per_year.csv')

# 3. Qatar 2022 team features
qatar_team_features.to_csv(OUTPUT_PATH + 'qatar_2022_team_features.csv', index=False)
print('✅ Saved: qatar_2022_team_features.csv')

# 4. Player Impact Scores
team_pis_df.to_csv(OUTPUT_PATH + 'team_player_impact_scores.csv', index=False)
print('✅ Saved: team_player_impact_scores.csv')

players[['player', 'team', 'position', 'minutes_90s',
          'pis_normalized', 'goals', 'assists', 'xg_per90']].to_csv(
    OUTPUT_PATH + 'player_impact_scores.csv', index=False
)
print('✅ Saved: player_impact_scores.csv')

print('\n🎉 Day 1 complete! All feature files saved to Drive.')
print('   Ready for Day 2: Model Training & Prediction Pipeline')

✅ Saved: ml_matches_features.csv
✅ Saved: elo_final_ratings.csv
✅ Saved: elo_entry_per_year.csv
✅ Saved: qatar_2022_team_features.csv
✅ Saved: team_player_impact_scores.csv
✅ Saved: player_impact_scores.csv

🎉 Day 1 complete! All feature files saved to Drive.
   Ready for Day 2: Model Training & Prediction Pipeline


## 7. Quick Insights — LinkedIn Post Material 📊

These cells generate visuals specifically for sharing.

In [ ]:
# ── Fix kaleido version conflict ───────────────────────────────────────────
!pip install -q kaleido==0.2.1

In [ ]:
# ── Visual 1: WC Champions History ───────────────────────────────────────
champion_counts = world_cups['Champion'].value_counts().reset_index()
champion_counts.columns = ['country', 'titles']

fig = px.bar(
    champion_counts,
    x='country', y='titles',
    color='titles',
    color_continuous_scale='Greens',
    title='🏆 World Cup Champions — All Time (1930–2022)',
    labels={'titles': 'Titles', 'country': ''},
    text='titles',
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickangle=-30,
    coloraxis_showscale=False,
    title_font_size=18
)
fig.show()
# Save for LinkedIn
fig.write_image(OUTPUT_PATH + 'viz_champions_history.png', width=900, height=500, scale=2)

In [ ]:
# ── Visual 2: Elo Rankings entering WC 2026 ───────────────────────────────
# Filter to likely WC 2026 participants (use WC 2022 teams as proxy)
wc2022_teams = team_data['team'].tolist()
elo_2026_proxy = elo_df[elo_df['team'].isin(wc2022_teams)].head(20)

fig = px.bar(
    elo_2026_proxy.sort_values('elo_rating'),
    x='elo_rating', y='team',
    orientation='h',
    color='elo_rating',
    color_continuous_scale='Blues',
    title='Elo Ratings — WC 2022 Participants (post-tournament)',
    labels={'elo_rating': 'Elo Rating', 'team': ''},
    template='plotly_white'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()
fig.write_image(OUTPUT_PATH + 'viz_elo_rankings.png', width=900, height=600, scale=2)

In [ ]:
# ── Visual 3: Player Impact Score bubble chart ─────────────────────────────
# (Already computed above — just add WC 2022 result context)

# Add whether team reached knockouts
knockout_teams = ['Argentina', 'France', 'Croatia', 'Morocco',
                   'Netherlands', 'England', 'Brazil', 'Portugal']

team_pis_viz = team_pis_df.copy()
team_pis_viz['reached_knockout'] = team_pis_viz['team'].isin(knockout_teams)

fig = px.scatter(
    team_pis_viz,
    x='pis_mean', y='pis_top3',
    text='team',
    size='pis_sum',
    color='reached_knockout',
    color_discrete_map={True: '#1f77b4', False: '#aec7e8'},
    title='Player Impact Score vs QF Qualification — Qatar 2022<br>Did individual quality predict knockout stage?',
    labels={
        'pis_mean': 'Avg Player Impact (Starting XI)',
        'pis_top3': 'Avg Impact — Top 3 Players',
        'reached_knockout': 'Reached QF'
    },
    template='plotly_white'
)
fig.update_traces(textposition='top center', textfont_size=8)
fig.show()
fig.write_image(OUTPUT_PATH + 'viz_player_impact_score.png', width=900, height=600, scale=2)

print('📸 3 LinkedIn-ready charts saved to Drive!')
print('\n🔑 Key insight: does high PIS correlate with reaching knockouts?')
correlation = team_pis_viz['pis_mean'].corr(team_pis_viz['reached_knockout'].astype(int))
print(f'   Correlation PIS_mean vs knockout qualification: {correlation:.3f}')

📸 3 LinkedIn-ready charts saved to Drive!

🔑 Key insight: does high PIS correlate with reaching knockouts?
   Correlation PIS_mean vs knockout qualification: 0.399


---

## ✅ Day 1 Complete — Summary

### What we built

| Output File | Description | Used In |
|-------------|-------------|--------|
| `ml_matches_features.csv` | 964 matches with Elo + win rates | Day 2: Model training |
| `elo_final_ratings.csv` | Current Elo per team | Day 2: Pre-tournament features |
| `elo_entry_per_year.csv` | Elo at start of each WC | Day 2: Historical validation |
| `qatar_2022_team_features.csv` | 32 teams × 40+ features | Day 2: Modern football signals |
| `team_player_impact_scores.csv` | PIS aggregated per team | Day 2: Player quality signal |
| `player_impact_scores.csv` | Individual PIS (680 players) | Dashboard: player cards |
| `viz_champions_history.png` | Chart: WC titles all time | LinkedIn post 1 |
| `viz_elo_rankings.png` | Chart: Elo entering WC 2026 | LinkedIn post 2 |
| `viz_player_impact_score.png` | Bubble: PIS vs results | LinkedIn post 3 |

### Key findings
- **Elo difference** is the single most predictive feature
- Group stage has ~25% draw rate vs ~0% in knockouts (model needs to learn this split)
- **Player Impact Score** correlates with knockout qualification
- xG data only available for 2018/2022 — use with care for historical model

### Day 2 plan
1. Train baseline model (Logistic Regression + LightGBM)
2. Validate with leave-one-tournament-out (Qatar 2022 as test set)
3. Build WC 2026 team feature matrix
4. Generate pre-tournament group predictions
5. Add LLM narrative layer (LangChain + Gemini)